# Module 06: Electronic Displays
**AE2.021 · USTH: Aircraft Digital Electronic & Computer Systems**

This notebook recomputes the module's numbers in code: pixel size from visual acuity, raster scan frequencies, 7-segment decoding (with Floyd's minimal expressions via sympy), a 74HC47 zero-suppression simulation, and checks of the numbers on the ECAM slides (fuel, ISA differential pressure, time to waypoint). It ends by cross-checking Tooley's printed answer key (read directly from scanned Appendix 3 images) against the questions that can be computed.

## 1. Visual acuity and pixel size
🎯 **What question does it answer?** At distance D, how small must a pixel be for the human eye no longer to resolve it (slides 5 and 6 of the lecture).

In [1]:
import math
theta = (1/60) * math.pi / 180            # 1 arcminute -> radians
print(f"theta = {theta:.4e} rad")
assert abs(theta - 2.91e-4) < 0.01e-4

lecture = {1: 0.29, 3: 0.87, 5: 1.45, 10: 2.91, 20: 5.82}   # table on slide 6 (mm)
for D, mm_slide in lecture.items():
    h_small = D * theta * 1000                         # h ~ D*theta
    h_exact = 2 * D * math.tan(theta / 2) * 1000       # more accurate
    print(f"D = {D:>2} m  h(D*theta) = {h_small:.3f} mm   h(2D tan(theta/2)) = {h_exact:.3f} mm   slide: {mm_slide}")
    assert abs(h_small - mm_slide) < 0.006

for name, (w, h) in {"HD": (1280, 720), "Full HD": (1920, 1080), "4K UHD": (3840, 2160)}.items():
    print(f"{name:8s} {w}x{h} = {w*h:>9,d} pixel = {w*h/1e6:.2f} MP")
assert round(1280*720/1e6, 1) == 0.9 and round(1920*1080/1e6, 1) == 2.1 and round(3840*2160/1e6, 1) == 8.3
assert 3840*2160 == 4 * 1920*1080               # 4K = 4 lan Full HD

theta = 2.9089e-04 rad
D =  1 m  h(D*theta) = 0.291 mm   h(2D tan(theta/2)) = 0.291 mm   slide: 0.29
D =  3 m  h(D*theta) = 0.873 mm   h(2D tan(theta/2)) = 0.873 mm   slide: 0.87
D =  5 m  h(D*theta) = 1.454 mm   h(2D tan(theta/2)) = 1.454 mm   slide: 1.45
D = 10 m  h(D*theta) = 2.909 mm   h(2D tan(theta/2)) = 2.909 mm   slide: 2.91
D = 20 m  h(D*theta) = 5.818 mm   h(2D tan(theta/2)) = 5.818 mm   slide: 5.82
HD       1280x720 =   921,600 pixel = 0.92 MP
Full HD  1920x1080 = 2,073,600 pixel = 2.07 MP
4K UHD   3840x2160 = 8,294,400 pixel = 8.29 MP


## 2. CRT raster scan (Tooley 11.1.3)

In [2]:
# Example 1: raster of 4 lines, Y = 50 Hz -> X = 4 * 50 = 200 Hz, one frame = 1/50 s = 20 ms
n_lines, f_y = 4, 50
f_x = n_lines * f_y
print("4 lines:", f_x, "Hz, frame =", 1000 / f_y, "ms")
assert f_x == 200 and 1000 / f_y == 20

# Example 2: 400 lines, Y = 100 Hz -> X = 40 kHz
n_lines, f_y = 400, 100
f_x = n_lines * f_y
print("400 lines:", f_x / 1000, "kHz, frame =", 1000 / f_y, "ms")
assert f_x == 40_000

# Number of colours of a pixel when each R,G,B channel has 256 levels (Tooley 11.3.3: 'more than 16 million colours')
print("Colours 256^3 =", 256**3)
assert 256**3 == 16_777_216 and 256**3 > 16_000_000

4 lines: 200 Hz, frame = 20.0 ms
400 lines: 40.0 kHz, frame = 10.0 ms
Colours 256^3 = 16777216


## 3. LEDs, 7-segment displays and Tooley's questions

In [3]:
SEG = {  # lit segments for each digit (standard 7-segment)
 0:"abcdef", 1:"bc", 2:"abged", 3:"abgcd", 4:"fgbc", 5:"afgcd", 6:"afgecd", 7:"abc", 8:"abcdefg", 9:"abcdfg"}
SEGSET = {k: frozenset(v) for k, v in SEG.items()}
def show(segs): return [k for k, v in SEGSET.items() if v == frozenset(segs)]

print("Segments a,b,c,d,g lit ->", show("abcdg"))       # Tooley Ch.11 Q5
print("BCD 1001 ->", int("1001", 2), "->", sorted(SEG[int('1001', 2)]))   # Tooley Ch.11 Q6
assert show("abcdg") == [3] and int("1001", 2) == 9
assert SEGSET[1] == frozenset("bc") and SEGSET[2] == frozenset("abged")   # Tooley section 11.2.2 (digits 1 and 2)

# Current-limiting resistor for a standard 5 mm LED (table 11.1: 10 mA, 2.0 V), 5 V supply (extra calculation of the module)
R = (5 - 2.0) / 0.010
print("R =", R, "ohm")
assert abs(R - 300) < 1e-9

Segments a,b,c,d,g lit -> [3]
BCD 1001 -> 9 -> ['a', 'b', 'c', 'd', 'f', 'g']
R = 300.0 ohm


## 4. Floyd Applied Logic: letter decoder (sympy)
🎯 The letters A, b, C, d, E use the segments of Table 4-14; outputs are active LOW (0 = lit); codes 0 to 9 are "don't care"; F is all off. From this, find the minimal expressions and check the printed answers of Problems 63, 65, 67, 69 (A-6).

In [4]:
from sympy import symbols, SOPform
from sympy.logic.boolalg import simplify_logic
H3, H2, H1, H0 = symbols("H3 H2 H1 H0"); V = [H3, H2, H1, H0]
on = {10: "abcefg", 11: "cdefg", 12: "adef", 13: "bcdeg", 14: "adefg", 15: ""}   # A b C d E F
dc = [[int(b) for b in format(c, "04b")] for c in range(10)]
def sop(seg, table=on):
    ones = [[int(b) for b in format(c, "04b")] for c in table if seg not in table[c]]   # output = 1 when the segment is OFF
    return SOPform(V, ones, dc)
def cost(e):
    terms = e.args if e.func.__name__ == "Or" else [e]
    ands, inv = {}, set()
    for t in terms:
        lits = t.args if t.func.__name__ == "And" else [t]
        if len(lits) > 1: ands[len(lits)] = ands.get(len(lits), 0) + 1
        inv |= {str(l.args[0]) for l in lits if l.func.__name__ == "Not"}
    return dict(AND=ands, OR=len(terms) if len(terms) > 1 else 0, inverters=len(inv))
res = {s: sop(s) for s in "abcdefg"}
for s, e in res.items(): print(s, "=", e, cost(e))

# Printed answer of Problem 65 (A-6): d = AND2 + AND3 + OR2 + 2 inverters; e = AND3; f = AND2; g = AND2 + AND3 + OR2 + 2 inverters
assert cost(res["d"]) == dict(AND={3: 1, 2: 1}, OR=2, inverters=2)
assert cost(res["e"]) == dict(AND={3: 1}, OR=0, inverters=0)
assert cost(res["f"]) == dict(AND={2: 1}, OR=0, inverters=0)
assert cost(res["g"]) == dict(AND={3: 1, 2: 1}, OR=2, inverters=2)
assert str(res["a"]) == "H0"                      # Floyd: segment a connects straight to H0
print("Matches the printed answer of Problem 65 and segment a = H0.")

a = H0 {'AND': {}, 'OR': 0, 'inverters': 0}
b = (H0 & H1) | (H2 & ~H0) {'AND': {2: 2}, 'OR': 2, 'inverters': 1}
c = (H1 & H2) | (~H0 & ~H1) {'AND': {2: 2}, 'OR': 2, 'inverters': 2}
d = (H0 & H1 & H2) | (~H0 & ~H2) {'AND': {3: 1, 2: 1}, 'OR': 2, 'inverters': 2}
e = H0 & H1 & H2 {'AND': {3: 1}, 'OR': 0, 'inverters': 0}
f = H0 & H2 {'AND': {2: 1}, 'OR': 0, 'inverters': 0}
g = (H0 & H1 & H2) | (~H0 & ~H1) {'AND': {3: 1, 2: 1}, 'OR': 2, 'inverters': 2}
Matches the printed answer of Problem 65 and segment a = H0.


In [5]:
# Problem 67: add the letter F (F lights a,e,f,g). Check the printed circuit of Figure P-19: a = H3.~H2.H1 + ~H1.H0
on67 = dict(on); on67[15] = "aefg"
need = {c: (0 if "a" in on67[c] else 1) for c in on67}          # a = 0 means lit
printed = lambda h3, h2, h1, h0: (h3 & (1 - h2) & h1) | ((1 - h1) & h0)
mine = lambda h3, h2, h1, h0: (h0 & (1 - h1)) | (h0 & (1 - h2))
bad = []
for c, v in need.items():
    h = [int(b) for b in format(c, "04b")]
    if printed(*h) != v: bad.append(format(c, "04b"))
    assert mine(*h) == v
print("The printed matrix is wrong at:", bad)
assert bad == ["1010"]                 # wrong only at the letter A: segment a of the letter A is not lit
print("Self-designed circuit a = H0.~H1 + H0.~H2 is correct on all 6 letters:", all(mine(*[int(b) for b in format(c, '04b')]) == v for c, v in need.items()))

The printed matrix is wrong at: ['1010']
Self-designed circuit a = H0.~H1 + H0.~H2 is correct on all 6 letters: True


In [6]:
# Problem 69: invalid-code detector X = H3.(H1 + H2): 1 for 1010..1111, 0 for 0000..1001
for c in range(16):
    h3, h2, h1, h0 = [int(b) for b in format(c, "04b")]
    x = h3 & (h1 | h2)
    assert x == (1 if c >= 10 else 0)
print("Problem 69: X = H3.(H1 + H2) is correct on all 16 codes.")
print(simplify_logic(H3 & (H1 | H2), "dnf"))

Problem 69: X = H3.(H1 + H2) is correct on all 16 codes.
(H1 & H3) | (H2 & H3)


## 5. 74HC47: zero suppression and lamp test (Floyd 6-5)

In [7]:
def show_digits(digits, leading=True):
    """Simulate the RBI/RBO chain of 74HC47s: leading (or trailing) zeros are blanked until a non-zero digit is met."""
    out = []; suppress = True
    seq = digits if leading else digits[::-1]
    for d in seq:
        if suppress and d == 0:
            out.append(" ")         # RBI = 0 and code 0000 -> all segments off, RBO = 0
        else:
            suppress = False; out.append(str(d))
    return "".join(out if leading else out[::-1])

# Floyd example: 030.080 displays as 30.08 (the necessary zeros in the middle and before the decimal point are kept)
print(repr(show_digits([0, 0, 3, 0], leading=True)))     # 0030 -> '  30'  (hinh 6-35a)
print(repr(show_digits([0, 5, 7, 0, 0], leading=False))) # 05700: only the TRAILING zeros are blanked -> '057  '
assert show_digits([0, 0, 3, 0], leading=True) == "  30"
assert show_digits([0, 5, 7, 0, 0], leading=False) == "057  "
assert show_digits([0, 3, 0, 0, 8, 0], leading=True) == " 30080"   # 030080: only the leading zero is blanked, the necessary zeros are kept
print("OK")

'  30'
'057  '
OK


## 6. Numbers on the lecture's ECAM slides
🎯 Independent checks of some numbers: total fuel (slide 32), ISA cabin differential pressure (slide 29), time to waypoint (slides 22 and 24), ACARS channels (Tooley table 15.1).

In [8]:
# 6.1 Fuel: sum of the tanks compared with the displayed FOB
off = [686, 2742, 5852, 2741, 686]; on_ = [106, 2742, 4902, 2741, 106]
print("Pumps OFF:", sum(off), "(FOB on slide 12707)"); print("Pumps ON:", sum(on_), "(FOB on slide 10598)")
assert sum(off) == 12707 and sum(on_) == 10597 and abs(sum(on_) - 10598) <= 1   # 1 kg difference = rounding of each tank

Pumps OFF: 12707 (FOB on slide 12707)
Pumps ON: 10597 (FOB on slide 10598)


In [9]:
# 6.2 Cabin differential pressure at FL140 under the ISA standard atmosphere (slide 29: dP = 2.9 psi, CAB ALT = 2870 ft)
def p_isa_psi(h_ft):
    return 14.696 * (1 - 6.8756e-6 * h_ft) ** 5.2559        # troposphere, h < 36089 ft
p_amb = p_isa_psi(14000); p_cab = p_isa_psi(2870)
dP = p_cab - p_amb
print(f"p(14000 ft) = {p_amb:.2f} psi, p(2870 ft) = {p_cab:.2f} psi, dP = {dP:.2f} psi (slide says 2.9)")
assert abs(p_amb - 8.63) < 0.02 and abs(p_cab - 13.24) < 0.02 and abs(dP - 4.6) < 0.05
T_isa = 15 - 1.98 * 14                                       # ~ -12.7 C: the slide shows SAT = -13 C (consistent with ISA)
print(f"ISA SAT at FL140 = {T_isa:.1f} C")
assert abs(T_isa - (-13)) < 0.5
# The ISA deviation cannot explain it: dP = 2.9 psi with CAB ALT 2870 ft needs an outside pressure = p_cab - 2.9 ->
p_need = p_cab - 2.9
h_need = (1 - (p_need / 14.696) ** (1 / 5.2559)) / 6.8756e-6
print(f"With dP = 2.9 psi, the outside altitude must be about {h_need:.0f} ft (not 14000 ft)")

p(14000 ft) = 8.63 psi, p(2870 ft) = 13.23 psi, dP = 4.60 psi (slide says 2.9)
ISA SAT at FL140 = -12.7 C
With dP = 2.9 psi, the outside altitude must be about 9424 ft (not 14000 ft)


In [10]:
# 6.3 Time to the waypoint on the ND: 37 NM at a ground speed of 257 kt
t_min = 37 / 257 * 60
print(f"t = {t_min:.1f} min  (the slide records '23 33' as 23 min 33 s, or '2 33' on another slide)")
assert abs(t_min - 8.6) < 0.05
# 6.4 ACARS channels (Tooley table 15.1) all lie in 129-137 MHz
ch = [129.125, 130.025, 130.450, 131.125, 131.475, 131.525, 131.550, 131.725, 136.900]
assert all(129 <= f <= 137 for f in ch) and len(ch) == 9
print("9 ACARS channels in the 129-137 MHz band: OK")

t = 8.6 min  (the slide records '23 33' as 23 min 33 s, or '2 33' on another slide)
9 ACARS channels in the 129-137 MHz band: OK


## 7. Cross-checking Tooley's printed answer key
The key was read directly from the scanned images of Appendix 3 (A.1 p.376, A.11 p.380, A.15 p.381). For questions that can be computed, the notebook recomputes and compares; Chapter 11 Q4 is a genuine printing error.

In [11]:
KEY_CH1 = "b a c a a b b b a a a a b b a c c b a a c b b a a c b c".split()   # questions 1..28 (A.1)
KEY_CH11 = "c b a a b c b a c b c c b a c a".split()                             # questions 1..16 (A.11)
KEY_CH15 = "c a c b c c b a c a b b a b".split()                                 # questions 1..14 (A.15)
assert len(KEY_CH1) == 28 and len(KEY_CH11) == 16 and len(KEY_CH15) == 14

# Computable questions of chapter 11
assert KEY_CH11[4] == "b"         # Q5: a,b,c,d,g -> '3' is option (b) among (a)2 (b)3 (c)5
assert KEY_CH11[5] == "c"         # Q6: BCD 1001 = 9 is option (c) among (a)7 (b)8 (c)9
assert KEY_CH11[0] == "c"         # Q1: 635 nm is option (c) among (a)475 (b)525 (c)635 (table 11.1)
assert KEY_CH11[13] == "a"        # Q14: 0.03 A is (a) among (a)0.03 (b)0.3 (c)3 (table 11.1: 30-40 mA)
assert KEY_CH11[12] == "b"        # Q13: ramp is (b)
# Questions 2 and 4 must be consistent (the eye is most sensitive at ~550 nm = green light)
q2_peak_nm = {"a": 450, "b": 550, "c": 650}[KEY_CH11[1]]
q4_choice = {"a": "red", "b": "blue", "c": "green"}[KEY_CH11[3]]
print("Q2 (printed key):", q2_peak_nm, "nm  ->", "green" if 520 <= q2_peak_nm <= 570 else "other")
print("Q4 (printed key):", q4_choice, " -> CONTRADICTS Q2: the correct answer is green = (c)")
assert 520 <= q2_peak_nm <= 570 and q4_choice != "green"        # confirms the misprint of the answer key
print("Chapter 1: 28 questions in total, question 20 ('secondary heading information') is left out: printed key (a) gyro, not self-checked so not put on the site.")

Q2 (printed key): 550 nm  -> green
Q4 (printed key): red  -> CONTRADICTS Q2: the correct answer is green = (c)
Chapter 1: 28 questions in total, question 20 ('secondary heading information') is left out: printed key (a) gyro, not self-checked so not put on the site.


#### 📤 Takeaway
Every number that can be recomputed matches the lecture and the books, except the three points flagged in the module: ΔP 2.9 psi at FL140 (computes to 4.6 psi), the "23 33" label (8.6 minutes), and the wrong printed answer key in Tooley Chapter 11 question 4. Floyd Problem 67's printed answer is wrong at code 1010 (the letter A).